In [ ]:
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.9/213.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.9/211.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 10.0 MB/s eta 0:00:00


In [ ]:
# @title
# 필요한 패키지 설치
!pip install -q "sdv>=1.13.0" openpyxl

# 버전 확인 (참고)
import sdv, pandas as pd
print("SDV version:", sdv.__version__)

# --- (1) 메타데이터 정의: 모두 범주형 컬럼 ---
from sdv.metadata import SingleTableMetadata

cols = ["성별", "연령대", "학력", "근무지역", "담당 영유아와 상호작용에서의 어려움", "담당 영유아 부모와의 관계에서의 어려움",
        "보육프로그램 운영에서의 어려움", "행정_사무 등의 업무처리에서의 어려움", "보육교직원과 관계에서의 어려움", "원장과의 관계에서의 어려움"]
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype="object") for c in cols})
)
# 모두 카테고리형으로 강제 지정
for c in cols:
    metadata.update_column(
        column_name=c,
        sdtype="categorical"
    )

# --- (2) 원본 로드 ---
import pandas as pd
df = pd.read_excel("/content/sample_data/10.보육교사 특성별 근무 제약사항 현황(원본).xlsx", sheet_name=0)
df = df[cols].copy()
for c in cols:
    df[c] = df[c].astype("string")  # 범주형 취급

# --- (3) CTGAN 학습 & 1000행 생성 ---
from sdv.single_table import CTGANSynthesizer

synth = CTGANSynthesizer(
    metadata,
    epochs=30,         # 가벼운 예시
    batch_size=64,
    pac=1,
    verbose=True
)
synth.fit(df)

synthetic = synth.sample(num_rows=1000)

# --- (4) 가장 단순한 안전성/유용성 지표 ---
import numpy as np
from scipy.stats import entropy
from pathlib import Path
import json

def single_out_rate(original: pd.DataFrame, s: pd.DataFrame) -> float:
    cols_fixed = list(original.columns)
    org_keys = original[cols_fixed].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[cols_fixed].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())

def jsd(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p, m) + entropy(q, m))

# 안전성(단순 일치율)
safety_single_out = single_out_rate(df, synthetic)

# 유용성(컬럼별 범주 분포 JSD 평균)
jsd_by_col = {}
for c in cols:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, fill_value=0).values,
                              vs.reindex(idx, fill_value=0).values))
utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))

# --- (5) 저장 ---
Path("/content/sample_data/").mkdir(parents=True, exist_ok=True)
synthetic.to_csv("/content/sample_data/synthetic_data.csv", index=False)
with pd.ExcelWriter("/content/sample_data/synthetic_data.xlsx") as w:
    synthetic.to_excel(w, index=False)

report = {
    "safety": {"single_out_rate": safety_single_out},
    "utility": {"jsd_by_column": jsd_by_col, "jsd_mean": utility_jsd_mean},
    "notes": "가장 단순한 안전성/유용성 지표로 점검한 원론적 예시"
}
with open("/content/sample_data/evaluation_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Single-Out Rate:", safety_single_out)
print("Mean JSD:", utility_jsd_mean)
print("Saved to /content/sample_data/")


SDV version: 1.27.0


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (0.49) | Discrim. (-0.09): 100%|██████████| 30/30 [00:07<00:00,  4.21it/s]


Single-Out Rate: 0.026
Mean JSD: 0.008236913886463696
Saved to /content/sample_data/


In [ ]:
# @title
# 내 공부용
# 혼합형(범주형 + 수치형) 테이블 합성: CTGAN(SDV 1.x)

# 1) 원본 로드(xlsx)
# 2) 메타데이터: 범주형=Categorical, 수치형=Numerival
# 3) CTGAN 학습 & 샘플링
# 4) 안전성=Single-Out Rate(수치형은 공정비교 위해 binning 후 일치율)
# 5) 유용성=JSD(범주형=빈도, 수치형=히스토그램)
# 6) 저장(csv/xlsx/json)

# 설치 필요시: !pip install -q "sdv>=1.13.0"openpyxl

# ----------------------------------------------------------------------
# 모듈 임포트
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import entropy # JSD 계산을 위해 엔트로피 함수 import
import sdv # 합성데이터 생성 라이브러리
# -----------------------------------------------------------------------
# (1) 기본 설정: 경로 및 하이퍼파라미터 정의
# 범주형(Categorical)으로 취급할 컬럼 목록
CAT_COLS = ["성별", "연령대", "가입연월"]
# 수치형(Numerical)으로 취급할 컬럼 목록
NUM_COLS = ["장바구니 품목수", "가입후_경과월"]
# 전체 컬럼 목록
ALL_COLS = CAT_COLS + NUM_COLS

# 입력데이터(엑셀 파일)경로
INPUT_XLSX = "/content/sample_data/9.온라인면세점 회원 활동 현황(합성용).xlsx"
# 결과물(합성데이터, 리포트)을 저장할 디렉토리 경로
OUTPUT_DIR = "/content/sample_data/"

# CTGAN 모델 학습 에포크(전체 데이터 반복 학습 횟수)
EPOCHS = 50
# 학습 시 한번에 처리할 데이터 샘플의 수
BATCH_SIZE = 64
# Packing-Aware Conditioning(PAC) 파라미터. 소규모 데이터셋에서 배치 크기 관련 문제를 방지
PAC = 1

# -----------------------------------------------------------------------
# (2) 메타데이터 정의(SDV 1.x 방식)
# SDV는 데이터의 타입과 제약조건 등을 정의하는 '메타데이터'를 기반으로 작동함
from sdv.metadata import SingleTableMetadata

# 단일 테이블용 메타데이터 객체 생성
metadata = SingleTableMetadata()

# 먼저 빈 데이터프레임 구조로 스키마를 감지시킨 후, 각 컬럼의 타입을 수동으로 지정
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype='object') for c in ALL_COLS})
)
# CAT_COLS 목록에 있는 컬럼들의 데이터 타입(sdtype)을 'categorical'로 설정
for c in CAT_COLS:
    metadata.update_column(column_name=c, sdtype='categorical')
# NUM_COLS 목록에 있는 컬럼들의 데이터 타입(sdtype)을 'numerical'로 설정
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype='numerical')

# ------------------------------------------------------------------------
# (3) 원본데이터 로드 및 전처리
# 엑셀 파일의 첫 번째 시트를 데이터프레임으로 로드
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
# 정의된 컬럼만 선택하고, 이후 수정을 위해 복사본을 생성
df = df[ALL_COLS].copy()

# 데이터 타입을 메타데이터와 일치시키기 위해 캐스팅 수행
# 범주형 컬럼은 문자열(string) 타입으로 변환
for c in CAT_COLS:
  df[c] = df[c].astype("string")
# 수치형 컬럼은 숫자(numeric) 타입으로 변환. 숫자로 변환 불가능한 값은 결측치(NaN)로 처리
for c in NUM_COLS:
  df[c] = pd.to_numeric(df[c], errors='coerce')

# 방어적 코딩: 수치형 컬럼에 결측치가 있다면 중앙값(median)으로 대체
for c in NUM_COLS:
  if df[c].isna().any():
    df[c] = df[c].fillna(df[c].median())

# 로드된 데이터의 형태(shape)와 각 컬럼의 데이터 타입(dtype)을 출력
print("[LOAD] shape", df.shape, "| dtypes:",{c:str(df[c].dtype) for c in df.columns})

# -------------------------------------------------------------------------
# (4) CTGAN 모델 학습 및 데이터 샘플링
from sdv.single_table import CTGANSynthesizer

# CTGAN 신디사이저(Synthesizer) 객체 생성 및 설정
synth = CTGANSynthesizer(
    metadata, # 위에서 정의한 메타데이터
    epochs=EPOCHS, # 학습 에포크
    batch_size=BATCH_SIZE, # 배치 크기
    pac=PAC, # PAC 파라미터
    verbose=True # 학습 진행 상황 출력
)
# 원본데이터(df)를 사용하여 CTGAN 모델 학습
synth.fit(df)

# 학습된 모델로부터 1000개의 합성 데이터 행(row)을 생성
synthetic = synth.sample(num_rows=1000)

# 생성된 데이터의 타입 일관성을 위해 후처리
# 범주형은 string으로, 수치형은 numeric으로 명확히 변환
for c in CAT_COLS:
  synthetic[c] = synthetic[c].astype("string")
for c in NUM_COLS:
  synthetic[c] = pd.to_numeric(synthetic[c], errors="coerce")

# ---------------------------------------------------------------------
# (5) 평가 지표 계산을 위한 유틸리티 함수 정의
def jsd(p,q,eps=1e-12):
  """
  Jensen-Shannon Divergence(JSD)를 계산. 두 확률 분포의 유사도를 측정.
  0에 가까울수록 두 분포가 유사함을 의미.
  """
  # 입력 배열을 float 타입으로 변환하고, log(0)을 피하기 위해 작은 값(epsilon)을 더함
  p = np.asarray(p, dtype=float) + eps
  q = np.asarray(q, dtype=float) + eps
  # 각 분포를 정규화하여 확률의 합이 1이 되도록 함
  p = p/p.sum(); q = q/q.sum()
  # 두 분포의 평균 분포 계산
  m = 0.5*(p+q)
  # JSD 계산: (KL(p,m) + KL(q,m)) / 2
  return 0.5*(entropy(p,m) + entropy(q,m))

def make_matching_keys_with_binning(original:pd.DataFrame,
                                    synth:pd.DataFrame,
                                    cat_cols,
                                    num_cols,
                                    qbins:int=20):
  """
  안전성 평가(Single-Out Rate)를 위해 원본과 합성 데이터의 각 행을 고유한 '키'로 변환.
  - 범주형: 원래 값 그대로 사용
  - 수치형: 공정한 비교를 위해 원본 + 합성 데이터 전체를 기준으로 동일한 구간(bin)으로 이산화(discretize)
  """
  o = original.copy()
  s = synth.copy()

  # 수치형 컬럼을 공통 구간으로 나누기
  for c in num_cols:
    # 원본과 합성 데이터를 합쳐 전체 분포를 파악
    both = pd.concat([o[c],s[c]],axis=0)
    try:
      # 전체 분포를 기준으로 qbins+1개의 분위수 경계값 계산
      q = np.linspace(0,1,qbins+1)
      # 중복된 경계값을 제거하여 오류 방지
      edges = np.unique(np.nanquantile(both.dropna(),q))
      # 경계가 2개 미만(예: 모든 값이 동일)이면 구간화 대신 반올림한 값을 문자열로 사용
      if len(edges) < 3:
        o[c+"_BIN"] = o[c].round(0).astype("int64").astype("string")
        s[c+"_BIN"] = s[c].round(0).astype("int64").astype("string")
      else:
        # np.digitize를 사용해 각 값이 어느 구간에 속하는지 인덱스를 부여
        o[c+"_BIN"] = np.digitize(o[c],bins=edges[1:-1],right=True).astype("int64").astype("string")
        s[c+"_BIN"] = np.digitize(s[c],bins=edges[1:-1],right=True).astype("int64").astype("string")
    except Exception:
      # 구간화 중 예외 발생 시, 반올림 기반의 안전한 방식으로 대체
      o[c+"_BIN"] = o[c].round(0).astype("int64").astype("string")
      s[c+"_BIN"] = s[c].round(0).astype("int64").astype("string")

  # 매칭에 사용할 컬럼 목록(원본 범주형 + 구간화된 수치형)
  match_cols = list(cat_cols) + [c+"_BIN" for c in num_cols]

  # 각 행의 값들을 "||"로 연결하여 고유 키(문자열) 생성
  org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
  syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r,values), axis=1)
  return org_keys, syn_keys

def single_out_rate_binned(original:pd.DataFrame,
                           synth:pd.DataFrame,
                           cat_cols,
                           num_cols,
                           qbins=20) -> float:
  """
  공통 quantile bin 기준으로 수치형을 이산화하여 공정하게 행 일치율 계산
  수치형 데이터를 구간화(binning)하여 공정하게 Single-Out Rate를 계산.
  합성데이터의 행이 원본 데이터에 존재하는 비율을 의미. (개인정보 노출 위험 지표)
  """
  org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
  # 원본 키들을 중복 없는 집합(set)으로 만듦
  org_set = set(org_keys.values)
  # 합성 키가 원본 키 집합에 포함되는 비율을 계산
  return float(syn.keys.isin(org_set).mean())

# -----------------------------------------------------------------------------------------
# (6) 안전성 및 유용성 지표 계산
# 6-1) 안전성 평가: Single-Out Rate 계산
safety_single_out = single_out_rate_binned(df,synthetic,CAT_COLS,NUM_COLS,qbins=20)

# 6-2) 유용성 평가: 컬럼별 JSD 계산
jsd_by_col = {}

# 범주형 컬럼의 JSD 계산
for c in CAT_COLS:
  # 원본과 합성 데이터의 각 카테고리별 빈도(비율) 계산
  vr = df[c].astype(str).value_counts(normalize=True)
  vs = synthetic[c].astype(str).value_counts(normalize=True)
  # 두 데이터에 모두 존재하는 카테고리 목록 생성
  idx = vr.index.union(vs.index)
  # 한쪽에만 존재하는 카테고리는 0으로 채워서 JSD 계산
  jsd_by_col[c] = float(jsd(vr.reindex(idx,fill_value=0).values,vs.reindex(idx,fill_value=0).values))

# 수치형 컬럼의 JSD 계산
for c in NUM_COLS:
  # 공정한 비교를 위해 원본 + 합성 데이터의 전체 범위를 기준으로 동일한 히스토그램 구간(bin) 설정
  both = pd.concat([df[c],synthetic[c]],axis=0).dropna()
  # 값이 모두 같으면 분포가 동일하다고 보고 JSD를 0으로 처리
  if both.nunique() <= 1:
    jsd_by_col[c] = 0.0
  else:
    bins=20
    vmin,vmax = float(both.min()),float(both.max())
    edges=np.linspace(vmin,vmax,bins+1)
    # 동일한 경계(edges)를 사용하여 원본과 합성 데이터의 히스토그램(분포) 계산
    hr,_ = np.histogram(df[c].dropna(),bins=edges,density=True)
    hs,_ = np.histogram(synthetic[c].dropna(),bins=edges,density=True)
    # 두 히스토그램 간의 JSD 계산
    jsd_by_col[c] = float(jsd(hr,hs))

# 전체 컬럼의 평균 JSD 계산
utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))

# 평가 결과 출력
print(f"[SAFETY] Single-Out Rate (binned): {safety_single_out:.6f}")
print(f"[UTILITY] Mean JSD: {utility_jsd_mean:.6f}")
for k,v in jsd_by_col.items():
  print(f" -{k}:{v:.6f}")


# ----------------------------------------------------------------------------------------
# (7) 결과물 저장
# 출력 디렉토리 생성(없는 경우)
Path(OUTPUT_DIR).mkdir(parents=True,exist_ok=True)

# 합성 데이터를 CSV 파일로 저장
synthetic.to_csv(os.path.join(OUTPUT_DIR,"온라인면세점 회원 활동 현황(합성).csv"),index=False)
# 합성 데이터를 엑셀 파일로 저장
with pd.ExcelWriter(os.path.join(OUTPUT_DIR,"온라인면세점 회원 활동 현황(합성).xlsx")) as w:
  synthetic.to_excel(w,index=False)

# 실험 설정 및 평과 결과를 담은 리포트(딕셔너리) 생성
report = {
    "config":{
        "epochs":EPOCHS,
        "batch_size":BATCH_SIZE,
        "pac":PAC,
        "cat_cols":CAT_COLS,
        "num_cols":NUM_COLS
    },
    "safety":{"single_out_rate_binned":safety_single_out,"qbins":20},
    "utility":{"jsd_by_column":jsd_by_col,"jsd_mean":utility_jsd_mean},
    "notes":"혼합형 테이블용:수치형은 binning 기반으로 안전성 매칭, 히스토그램 기반으로 JSD 유용성 평가"
}

# 리포트를 JSON 파일로 저장(한글 깨짐 방지 'ensure_ascii=False')
with open(os.path.join(OUTPUT_DIR),"evaluation_report.json","w",encoding="utf-8") as f:
  json.dump(report,f,indent=2,ensure_ascii=False)

print("Saved_to:",OUTPUT_DIR)


SyntaxError: invalid syntax. Perhaps you forgot a comma? (ipython-input-2349878775.py, line 250)

소득분위 제약조건 적용 합성데이터 코드

In [ ]:
# @title
# =========================================================
# 혼합형(범주형+수치형) 합성: CTGAN(SDV 1.x)
#  - '소득분위'의 NA는 "평가기준 비적용"이라는 의미 → '소득분위_적용' 플래그 추가
#  - 안전성: Single-Out Rate (수치형 binning, NA 독립 버킷)
#  - 유용성: JSD (범주형=빈도, 수치형=히스토그램+NA 버킷)
#  - 샘플링: "오버샘플+수리 루프"로 항상 정확히 1,000행 맞춤
# ---------------------------------------------------------
# 필요시 설치:
#   !pip install -q "sdv>=1.13.0" openpyxl
# =========================================================

import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

import sdv
print("SDV version:", sdv.__version__)

# -----------------------------
# (1) 컬럼/경로/파라미터 설정
# -----------------------------
CAT_COLS = ["연령대", "성별", "주택소재지_자치구", "임대주택 유형", "일반_우선_특별", "주택면적", "당첨_예비"]  # 범주형
NUM_COLS = ["가족수", "소득분위"]  # 수치형(소득분위 NA=비적용)
ALL_COLS = CAT_COLS + NUM_COLS

INPUT_XLSX = "/content/sample_data/5.임대주택 당첨자 정보 현황(합성용).xlsx"
OUTPUT_DIR = "/content/sample_data/"

EPOCHS = 50
BATCH_SIZE = 64
PAC = 1

TARGET_ROWS = 1000       # 최종 목표 행 수
BATCH_GEN = 800          # 한 번에 뽑을 양(충분히 넉넉히)
MAX_TRIES = 10           # 보충 루프 최대 반복

# -----------------------------
# (2) 원본 로드 & 의미있는 NA 보존
# -----------------------------
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
df = df[ALL_COLS].copy()

# 타입 캐스팅
for c in CAT_COLS:
    df[c] = df[c].astype("string")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# '소득분위' NA는 "비적용" 의미 → 플래그 보존
df["소득분위_적용"] = pd.Series(
    np.where(df["소득분위"].isna(), "비적용", "적용"),
    index=df.index
).astype("string")

# 합성에 사용할 전체 컬럼(플래그 포함)
CAT_COLS_PLUS = CAT_COLS + ["소득분위_적용"]
ALL_COLS_PLUS = CAT_COLS_PLUS + NUM_COLS

print("[LOAD] shape:", df.shape)
print("[LOAD] 소득분위 NA 비율:", float(df["소득분위"].isna().mean()))

# -----------------------------
# (3) 메타데이터 정의 (SDV 1.x)
# -----------------------------
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
# 빈 데이터프레임으로 스키마 감지 후 수동 sdtype 지정
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype="object") for c in ALL_COLS_PLUS})
)
for c in CAT_COLS_PLUS:
    metadata.update_column(column_name=c, sdtype="categorical")
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype="numerical")

# -----------------------------
# (4) 합성 모델 학습
# -----------------------------
from sdv.single_table import CTGANSynthesizer

synth = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    pac=PAC,
    verbose=True
)
synth.fit(df[ALL_COLS_PLUS])

# -----------------------------
# (5) 일관성 수리 함수 + 오버샘플 루프
# -----------------------------
def repair_consistency(df_syn: pd.DataFrame) -> pd.DataFrame:
    """비즈니스 규칙 수리:
       - 비적용 → 소득분위=NA 강제
       - 적용인데 소득분위 NA → 제거(권장)
    """
    out = df_syn.copy()
    # 타입 정리
    for c in CAT_COLS_PLUS: out[c] = out[c].astype("string")
    for c in NUM_COLS:      out[c] = pd.to_numeric(out[c], errors="coerce")

    # 규칙 1: 비적용이면 NA 강제
    out.loc[out["소득분위_적용"] == "비적용", "소득분위"] = np.nan
    # 규칙 2: 적용인데 NA면 제거 (필요 시 치환 로직으로 대체 가능)
    bad = (out["소득분위_적용"] == "적용") & (out["소득분위"].isna())
    out = out.loc[~bad].copy()
    return out

kept = []
tries = 0
while sum(len(k) for k in kept) < TARGET_ROWS and tries < MAX_TRIES:
    tries += 1
    chunk = synth.sample(num_rows=BATCH_GEN)

    # 타입 정리
    for c in CAT_COLS_PLUS: chunk[c] = chunk[c].astype("string")
    for c in NUM_COLS:      chunk[c] = pd.to_numeric(chunk[c], errors="coerce")

    # 일관성 수리
    chunk = repair_consistency(chunk)
    kept.append(chunk)

synthetic = pd.concat(kept, ignore_index=True)
synthetic = synthetic.head(TARGET_ROWS)   # 정확히 1000행 맞추기

print("[SAMPLE] after repair rows:", len(synthetic))
print("[SAMPLE] 적용인데 NA 비율(should be 0):",
      float(((synthetic["소득분위_적용"]=="적용") & (synthetic["소득분위"].isna())).mean()))

# -----------------------------
# (6) 지표 함수들 (NA-aware)
# -----------------------------
def jsd(p, q, eps=1e-12):
    """Jensen–Shannon Divergence: 0에 가까울수록 분포 유사"""
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p, m) + entropy(q, m))

def make_matching_keys_with_binning(original: pd.DataFrame,
                                    synth: pd.DataFrame,
                                    cat_cols,
                                    num_cols,
                                    qbins: int = 20):
    """Single-Out Rate 계산용 키:
       - 범주형은 그대로
       - 수치형은 공통 quantile bin으로 이산화
       - NA는 '__NA__' 버킷
    """
    o = original.copy()
    s = synth.copy()
    NA_LABEL = "__NA__"

    for c in num_cols:
        both = pd.concat([o[c], s[c]], axis=0)
        try:
            q = np.linspace(0, 1, qbins + 1)
            edges = np.unique(np.nanquantile(both.dropna(), q))
        except Exception:
            edges = np.array([])

        if len(edges) < 3:
            # 상수열 등 엣지 부족 → 라운딩 문자열 + NA 라벨
            o[c + "_BIN"] = o[c].round(0).astype("Int64").astype("string")
            s[c + "_BIN"] = s[c].round(0).astype("Int64").astype("string")
            o.loc[o[c].isna(), c + "_BIN"] = NA_LABEL
            s.loc[s[c].isna(), c + "_BIN"] = NA_LABEL
        else:
            # digitize → Series로 감싼 뒤 string 변환
            o_bins = pd.Series(
                np.digitize(o[c].to_numpy(), bins=edges[1:-1], right=True),
                index=o.index
            )
            s_bins = pd.Series(
                np.digitize(s[c].to_numpy(), bins=edges[1:-1], right=True),
                index=s.index
            )
            o[c + "_BIN"] = o_bins.astype("int64").astype("string")
            s[c + "_BIN"] = s_bins.astype("int64").astype("string")
            o.loc[o[c].isna(), c + "_BIN"] = NA_LABEL
            s.loc[s[c].isna(), c + "_BIN"] = NA_LABEL

    match_cols = list(cat_cols) + [c + "_BIN" for c in num_cols]
    org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    return org_keys, syn_keys

def single_out_rate_binned(original: pd.DataFrame, synth: pd.DataFrame,
                           cat_cols, num_cols, qbins=20) -> float:
    org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())

# -----------------------------
# (7) 안전성 & 유용성 계산
# -----------------------------
# 안전성: 플래그 포함 + NA-aware binning
safety_single_out = single_out_rate_binned(
    df[ALL_COLS_PLUS], synthetic[ALL_COLS_PLUS],
    cat_cols=CAT_COLS_PLUS, num_cols=NUM_COLS, qbins=20
)

# 유용성: 범주형(플래그 포함)은 빈도 JSD, 수치형은 히스토그램 + NA 버킷
jsd_by_col = {}

# 범주형
for c in CAT_COLS_PLUS:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, fill_value=0).values,
                              vs.reindex(idx, fill_value=0).values))

# 수치형 (마지막 칸에 NA 버킷)
for c in NUM_COLS:
    both = pd.concat([df[c], synthetic[c]], axis=0)
    na_r = df[c].isna().mean()
    na_s = synthetic[c].isna().mean()

    if both.dropna().nunique() <= 1:
        pr = np.array([1 - na_r, na_r])
        ps = np.array([1 - na_s, na_s])
        jsd_by_col[c] = float(jsd(pr, ps))
    else:
        bins = 20
        vmin, vmax = float(both.min(skipna=True)), float(both.max(skipna=True))
        edges = np.linspace(vmin, vmax, bins + 1)
        hr, _ = np.histogram(df[c].dropna(), bins=edges, density=False)
        hs, _ = np.histogram(synthetic[c].dropna(), bins=edges, density=False)
        pr = np.append(hr, int(df[c].isna().sum())).astype(float)
        ps = np.append(hs, int(synthetic[c].isna().sum())).astype(float)
        pr = pr / (pr.sum() if pr.sum() > 0 else 1.0)
        ps = ps / (ps.sum() if ps.sum() > 0 else 1.0)
        jsd_by_col[c] = float(jsd(pr, ps))

utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))

print(f"[SAFETY] Single-Out Rate (binned, NA-aware): {safety_single_out:.6f}")
print(f"[UTILITY] Mean JSD (NA-aware): {utility_jsd_mean:.6f}")
for k, v in jsd_by_col.items():
    print(f"  - {k}: {v:.6f}")

# -----------------------------
# (8) 저장
# -----------------------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
synthetic.to_csv(os.path.join(OUTPUT_DIR, "synthetic_data.csv"), index=False)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "synthetic_data.xlsx")) as w:
    synthetic.to_excel(w, index=False)

report = {
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "pac": PAC,
        "cat_cols": CAT_COLS, "num_cols": NUM_COLS,
        "cat_cols_plus": CAT_COLS_PLUS,
        "target_rows": TARGET_ROWS, "batch_gen": BATCH_GEN, "max_tries": MAX_TRIES
    },
    "sampling": {
        "repair": {
            "rule1": "비적용 -> 소득분위=NA 강제",
            "rule2": "적용 & 소득분위=NA -> drop"
        },
        "final_rows": int(len(synthetic))
    },
    "safety": {"single_out_rate_binned_na_aware": float(safety_single_out), "qbins": 20},
    "utility": {"jsd_by_column": {k: float(v) for k, v in jsd_by_col.items()},
                "jsd_mean": float(utility_jsd_mean)},
    "notes": "의미있는 NA(비적용) 보존, NA-aware 지표, 오버샘플+수리 루프로 정확히 목표 행수 보장"
}
with open(os.path.join(OUTPUT_DIR, "evaluation_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved to:", OUTPUT_DIR)


SDV version: 1.27.0
[LOAD] shape: (1000, 10)
[LOAD] 소득분위 NA 비율: 0.416


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (0.12) | Discrim. (-0.31): 100%|██████████| 50/50 [00:26<00:00,  1.87it/s]


[SAMPLE] after repair rows: 1000
[SAMPLE] 적용인데 NA 비율(should be 0): 0.0
[SAFETY] Single-Out Rate (binned, NA-aware): 0.011000
[UTILITY] Mean JSD (NA-aware): 0.009050
  - 연령대: 0.003719
  - 성별: 0.001878
  - 주택소재지_자치구: 0.009226
  - 임대주택 유형: 0.004665
  - 일반_우선_특별: 0.000304
  - 주택면적: 0.006068
  - 당첨_예비: 0.007783
  - 소득분위_적용: 0.000247
  - 가족수: 0.009593
  - 소득분위: 0.047018
Saved to: /content/sample_data/


In [ ]:
# @title
# =========================================================
# 혼합형(범주형+수치형) 테이블 합성: CTGAN (SDV 1.x)
# - 학습: sin/cos 포함
# - 평가: 원본 항목 기준 (sin/cos 제외)
# ---------------------------------------------------------
# 필요시 사전 설치:
#   pip install "sdv>=1.13.0" openpyxl
# =========================================================

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

import sdv
print("SDV version:", sdv.__version__)

# -----------------------------
# 0) 설정
# -----------------------------
# 파일 경로
INPUT_XLSX = "/content/sample_data/9.온라인면세점 회원 활동 현황(합성용).xlsx"
OUTPUT_DIR = "/content/sample_data/"

# 학습 파라미터
EPOCHS = 50
BATCH_SIZE = 64
PAC = 1
SAMPLE_ROWS = 1000

# 재현성(선택)
SEED = 42
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except Exception:
    pass

# -----------------------------
# 1) 컬럼 세트 (중요)
# -----------------------------
# 학습용 전체 컬럼
CAT_COLS = ["성별", "연령대", "가입연월"]
NUM_COLS = ["장바구니 품목수", "가입후_경과월수", "가입월_sin", "가입월_cos"]
ALL_COLS = CAT_COLS + NUM_COLS

# ★ 평가용 컬럼 (sin/cos 제외)
CAT_COLS_EVAL = ["성별", "연령대", "가입연월"]
NUM_COLS_EVAL = ["장바구니 품목수", "가입후_경과월수"]

# -----------------------------
# 2) 메타데이터 정의
# -----------------------------
from sdv.metadata import SingleTableMetadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=pd.DataFrame({c: pd.Series(dtype="object") for c in ALL_COLS}))
for c in CAT_COLS:
    metadata.update_column(column_name=c, sdtype="categorical")
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype="numerical")

# -----------------------------
# 3) 데이터 로드/전처리
# -----------------------------
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
missing = [c for c in ALL_COLS if c not in df.columns]
if missing:
    raise ValueError(f"입력 파일에 필요한 컬럼이 없습니다: {missing}")

df = df[ALL_COLS].copy()

# 타입 캐스팅
for c in CAT_COLS:
    df[c] = df[c].astype("string")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

print("[LOAD] shape:", df.shape)
print("[LOAD] dtypes:", {c: str(df[c].dtype) for c in df.columns})

# -----------------------------
# 4) CTGAN 학습/샘플링
# -----------------------------
from sdv.single_table import CTGANSynthesizer
synth = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    pac=PAC,
    verbose=True,
    cuda=False  # 필요시 True
)
synth.fit(df)
synthetic = synth.sample(num_rows=SAMPLE_ROWS)

# 후처리(형 일관화)
for c in CAT_COLS:
    synthetic[c] = synthetic[c].astype("string")
for c in NUM_COLS:
    synthetic[c] = pd.to_numeric(synthetic[c], errors="coerce")

# -----------------------------
# 5) 평가 유틸
# -----------------------------
def jsd(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    return 0.5 * (entropy(p, m) + entropy(q, m))

def make_matching_keys_with_binning(original: pd.DataFrame,
                                    synth: pd.DataFrame,
                                    cat_cols, num_cols,
                                    qbins: int = 20):
    o = original.copy()
    s = synth.copy()

    for c in num_cols:
        both = pd.concat([o[c], s[c]], axis=0)
        try:
            q = np.linspace(0, 1, qbins + 1)
            edges = np.unique(np.nanquantile(both.dropna(), q))
            if len(edges) < 3:
                o[c + "_BIN"] = o[c].round(0).astype("Int64").astype("string")
                s[c + "_BIN"] = s[c].round(0).astype("Int64").astype("string")
            else:
                o[c + "_BIN"] = np.digitize(o[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
                s[c + "_BIN"] = np.digitize(s[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
        except Exception:
            o[c + "_BIN"] = o[c].round(0).astype("Int64").astype("string")
            s[c + "_BIN"] = s[c].round(0).astype("Int64").astype("string")

    match_cols = list(cat_cols) + [c + "_BIN" for c in num_cols]
    org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    return org_keys, syn_keys

def single_out_rate_binned(original: pd.DataFrame, synth: pd.DataFrame,
                           cat_cols, num_cols, qbins=20) -> float:
    org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())

# -----------------------------
# 6) 평가(★ sin/cos 제외한 컬럼만 사용)
# -----------------------------
safety_single_out = single_out_rate_binned(
    df, synthetic,
    CAT_COLS_EVAL, NUM_COLS_EVAL,
    qbins=20
)

jsd_by_col = {}
# 범주형 JSD
for c in CAT_COLS_EVAL:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, fill_value=0).values,
                              vs.reindex(idx, fill_value=0).values))
# 수치형 JSD
for c in NUM_COLS_EVAL:
    both = pd.concat([df[c], synthetic[c]], axis=0).dropna()
    if both.nunique() <= 1:
        jsd_by_col[c] = 0.0
    else:
        bins = 20
        vmin, vmax = float(both.min()), float(both.max())
        edges = np.linspace(vmin, vmax, bins + 1)
        hr, _ = np.histogram(df[c].dropna(), bins=edges, density=True)
        hs, _ = np.histogram(synthetic[c].dropna(), bins=edges, density=True)
        jsd_by_col[c] = float(jsd(hr, hs))

utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))

print(f"[SAFETY] Single-Out Rate (binned, eval-only): {safety_single_out:.6f}")
print(f"[UTILITY] Mean JSD (eval-only): {utility_jsd_mean:.6f}")
for k, v in jsd_by_col.items():
    print(f"  - {k}: {v:.6f}")

# -----------------------------
# 7) 저장
# -----------------------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
syn_csv  = os.path.join(OUTPUT_DIR, "9.온라인면세점 회원 활동 현황(합성).csv")
syn_xlsx = os.path.join(OUTPUT_DIR, "9.온라인면세점 회원 활동 현황(합성).xlsx")
rep_json = os.path.join(OUTPUT_DIR, "evaluation_report.json")

synthetic.to_csv(syn_csv, index=False)
with pd.ExcelWriter(syn_xlsx) as w:
    synthetic.to_excel(w, index=False)

report = {
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "pac": PAC, "seed": SEED,
        "cat_cols_train": CAT_COLS, "num_cols_train": NUM_COLS,
        "cat_cols_eval":  CAT_COLS_EVAL, "num_cols_eval": NUM_COLS_EVAL
    },
    "safety": {"single_out_rate_binned": safety_single_out, "qbins": 20},
    "utility": {"jsd_by_column": jsd_by_col, "jsd_mean": utility_jsd_mean},
    "notes": "평가에는 sin/cos 파생변수 제외(원본 항목 기준)."
}
with open(rep_json, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved:", syn_csv)
print("Saved:", syn_xlsx)
print("Saved:", rep_json)


SDV version: 1.26.0
[LOAD] shape: (1138, 7)
[LOAD] dtypes: {'성별': 'string', '연령대': 'string', '가입연월': 'string', '장바구니 품목수': 'int64', '가입후_경과월수': 'int64', '가입월_sin': 'float64', '가입월_cos': 'float64'}


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:167: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:133: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (0.83) | Discrim. (-0.35): 100%|██████████| 50/50 [00:31<00:00,  1.59it/s]


[SAFETY] Single-Out Rate (binned, eval-only): 0.097000
[UTILITY] Mean JSD (eval-only): 0.013794
  - 성별: 0.000942
  - 연령대: 0.021404
  - 가입연월: 0.015856
  - 장바구니 품목수: 0.007027
  - 가입후_경과월수: 0.023740
Saved: /content/sample_data/9.온라인면세점 회원 활동 현황(합성).csv
Saved: /content/sample_data/9.온라인면세점 회원 활동 현황(합성).xlsx
Saved: /content/sample_data/evaluation_report.json


In [ ]:
# 정석 코드
# =========================================================
# 혼합형(범주형 + 수치형) 테이블 합성: CTGAN (SDV 1.x)
# 1) 원본 로드(xlsx)
# 2) 메타데이터: 범주형=Categorical, 수치형=Numerical
# 3) CTGAN 학습 & 샘플링
# 4) 안전성(단순) = Single-Out Rate (수치형은 공정비교 위해 binning 후 일치율)
# 5) 유용성(단순) = JSD (범주형=빈도, 수치형=히스토그램)
# 6) 저장(csv/xlsx/json)
# ---------------------------------------------------------
# 설치 필요시:
#   !pip install -q "sdv>=1.13.0" openpyxl
# =========================================================

import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

import sdv
print("SDV version:", sdv.__version__)

# --- (1) 컬럼 정의 ----------------------------------------------------------
# 예시: '월 임금총액'이 수치형이고 나머지는 범주형이라고 가정
CAT_COLS = ["성별", "학력", "직업분류", "최근인터넷이용시기", "인터넷이용빈도", "월평균가구소득", "거주지", ]   # 범주형
NUM_COLS = ["가구원연령"]                                       # 수치형
ALL_COLS = CAT_COLS + NUM_COLS

INPUT_XLSX = "/content/sample_data/9. 개인 인터넷 이용행태 정보.xlsx"
OUTPUT_DIR = "/content/sample_data/"
EPOCHS = 10
BATCH_SIZE = 200
PAC = 10  # 소규모 데이터에 안전(배치가 pac으로 나눠떨어지지 않아도 안전)

# --- (2) 메타데이터 정의 (SDV 1.x) ----------------------------------------
# SingleTableMetadata는 deprecated 경고가 나오지만 간단해서 계속 사용
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
# 스키마를 "비어있는 형태"로 감지한 뒤, 수동으로 sdtype 지정
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype="object") for c in ALL_COLS})
)
# 범주형 지정
for c in CAT_COLS:
    metadata.update_column(column_name=c, sdtype="categorical")
# 수치형 지정 (정수/실수 구분이 필요하면 여기서 더 세분화 가능)
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype="numerical")

# --- (3) 원본 로드 & 타입 캐스팅 ------------------------------------------
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
df = df[ALL_COLS].copy()

# 범주형 → 문자열(또는 category) / 수치형 → numeric
for c in CAT_COLS:
    df[c] = df[c].astype("string")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 결측치가 없다 했지만, 방어적으로 수치형 결측이 생기면 간단 대체(중앙값)
for c in NUM_COLS:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())

print("[LOAD] shape:", df.shape, "| dtypes:", {c: str(df[c].dtype) for c in df.columns})

# --- (4) CTGAN 학습 & 샘플링 -----------------------------------------------
from sdv.single_table import CTGANSynthesizer

synth = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    pac=PAC,
    verbose=True
)
synth.fit(df)

synthetic = synth.sample(num_rows=10000)

# 수치형은 float로, 범주형은 string으로 정리(일관성)
for c in CAT_COLS:
    synthetic[c] = synthetic[c].astype("string")
for c in NUM_COLS:
    synthetic[c] = pd.to_numeric(synthetic[c], errors="coerce")

# --- (5) 지표 유틸리티 함수 -------------------------------------------------
def jsd(p, q, eps=1e-12):
    """Jensen-Shannon Divergence: 0에 가까울수록 분포 유사"""
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p, m) + entropy(q, m))

def make_matching_keys_with_binning(original: pd.DataFrame,
                                    synth: pd.DataFrame,
                                    cat_cols,
                                    num_cols,
                                    qbins: int = 20):
    """
    Single-Out Rate 계산을 위해:
      - 범주형은 그대로 사용
      - 수치형은 원본+합성의 분포를 기준으로 '공통 quantile bin'으로 이산화
        -> 수치형을 직접 값으로 비교하는 불공정(미세 소수점 차이 등) 방지
    """
    o = original.copy()
    s = synth.copy()

    # 수치형 공통 bin 엣지 계산(원본+합성 합쳐서 quantile 기준)
    for c in num_cols:
        both = pd.concat([o[c], s[c]], axis=0)
        # 유효 값만으로 qcut 엣지 결정 (중복 엣지 문제 방지 위해 duplicates='drop')
        try:
            q = np.linspace(0, 1, qbins+1)
            edges = np.unique(np.nanquantile(both.dropna(), q))
            # 경계가 2개 미만(상수열 등)이면, 간단히 그대로 문자열 변환로 fallback
            if len(edges) < 3:
                o[c+"_BIN"] = o[c].round(0).astype("Int64").astype("string")
                s[c+"_BIN"] = s[c].round(0).astype("Int64").astype("string")
            else:
                # np.digitize 로 구간 부여 (오른쪽 닫힘 포함 조정)
                o[c+"_BIN"] = np.digitize(o[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
                s[c+"_BIN"] = np.digitize(s[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
        except Exception:
            # 예외 시 라운딩 기반 fallback
            o[c+"_BIN"] = o[c].round(0).astype("Int64").astype("string")
            s[c+"_BIN"] = s[c].round(0).astype("Int64").astype("string")

    # 매칭에 사용할 최종 컬럼 묶음(범주형 + 수치형BIN)
    match_cols = list(cat_cols) + [c+"_BIN" for c in num_cols]

    org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    return org_keys, syn_keys

def single_out_rate_binned(original: pd.DataFrame, synth: pd.DataFrame,
                           cat_cols, num_cols, qbins=20) -> float:
    """
    공통 quantile bin 기준으로 수치형을 이산화하여 공정하게 행 일치율 계산
    """
    org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())

# --- (6) 안전성(단순) & 유용성(단순) ----------------------------------------
# 6-1) 안전성: Single-Out Rate (수치형은 binning 기반 매칭)
safety_single_out = single_out_rate_binned(df, synthetic, CAT_COLS, NUM_COLS, qbins=20)

# 6-2) 유용성: JSD
#  - 범주형: 빈도 분포 JSD
#  - 수치형: 히스토그램(공통 엣지) JSD
jsd_by_col = {}

# 범주형
for c in CAT_COLS:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, fill_value=0).values,
                              vs.reindex(idx, fill_value=0).values))

# 수치형
for c in NUM_COLS:
    # 공통 엣지: 원본+합성 합쳐서 범위 파악 후 동일 엣지 사용
    both = pd.concat([df[c], synthetic[c]], axis=0).dropna()
    if both.nunique() <= 1:
        # 상수열이면 분포가 동일하다고 보고 0
        jsd_by_col[c] = 0.0
    else:
        bins = 20
        vmin, vmax = float(both.min()), float(both.max())
        edges = np.linspace(vmin, vmax, bins+1)
        hr, _ = np.histogram(df[c].dropna(), bins=edges, density=True)
        hs, _ = np.histogram(synthetic[c].dropna(), bins=edges, density=True)
        jsd_by_col[c] = float(jsd(hr, hs))

utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))

print(f"[SAFETY] Single-Out Rate (binned): {safety_single_out:.6f}")
print(f"[UTILITY] Mean JSD: {utility_jsd_mean:.6f}")
for k, v in jsd_by_col.items():
    print(f"  - {k}: {v:.6f}")

# --- (7) 저장 ---------------------------------------------------------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
synthetic.to_csv(os.path.join(OUTPUT_DIR, "synthetic_data.csv"), index=False)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "synthetic_data.xlsx")) as w:
    synthetic.to_excel(w, index=False)

report = {
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "pac": PAC,
        "cat_cols": CAT_COLS, "num_cols": NUM_COLS
    },
    "safety": {"single_out_rate_binned": safety_single_out, "qbins": 20},
    "utility": {"jsd_by_column": jsd_by_col, "jsd_mean": utility_jsd_mean},
    "notes": "혼합형 테이블용: 수치형은 binning 기반으로 안전성 매칭, 히스토그램 기반 JSD로 유용성 평가"
}
with open(os.path.join(OUTPUT_DIR, "evaluation_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved to:", OUTPUT_DIR)


SDV version: 1.38.1
[LOAD] shape: (37298, 8) | dtypes: {'성별': 'string', '학력': 'string', '직업분류': 'string', '최근인터넷이용시기': 'string', '인터넷이용빈도': 'string', '월평균가구소득': 'string', '거주지': 'string', '가구원연령': 'int64'}


/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.13/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-00.77) | Discrim. (-00.29): 100%|██████████| 10/10 [00:53<00:00,  5.39s/it]


[SAFETY] Single-Out Rate (binned): 0.193400
[UTILITY] Mean JSD: 0.021060
  - 성별: 0.010517
  - 학력: 0.012607
  - 직업분류: 0.015573
  - 최근인터넷이용시기: 0.037568
  - 인터넷이용빈도: 0.025222
  - 월평균가구소득: 0.030246
  - 거주지: 0.012758
  - 가구원연령: 0.023990
Saved to: /content/sample_data/


In [ ]:
# @title
# 캡쳐용

# 1) 메타데이터 정의
from sdv.metadata import SingleTableMetadata

metadata = SingleTableMetadata()
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype="object") for c in ALL_COLS})
)
for c in CAT_COLS:
    metadata.update_column(column_name=c, sdtype="categorical")
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype="numerical")

# 2) 원본 로드 & 타입 캐스팅
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
df = df[ALL_COLS].copy()

for c in CAT_COLS:
    df[c] = df[c].astype("string")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in NUM_COLS:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())


# 3) CTGAN 학습 & 샘플링
from sdv.single_table import CTGANSynthesizer

synth = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    pac=PAC,
    verbose=True
)
synth.fit(df)

synthetic = synth.sample(num_rows=1000)

for c in CAT_COLS:
    synthetic[c] = synthetic[c].astype("string")
for c in NUM_COLS:
    synthetic[c] = pd.to_numeric(synthetic[c], errors="coerce")






# 1) 유용성 검증(JSD)
def jsd(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p, m) + entropy(q, m))
# 범주형 컬럼
for c in CAT_COLS:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, 0).values, vs.reindex(idx, 0).values))
# 수치형 컬럼
for c in NUM_COLS:
    both = pd.concat([df[c], synthetic[c]], axis=0).dropna()
    if both.nunique() <= 1:
        jsd_by_col[c] = 0.0
    else:
        edges = np.linspace(both.min(), both.max(), 21)  # 20 bins
        hr, _ = np.histogram(df[c].dropna(), bins=edges, density=True)
        hs, _ = np.histogram(synthetic[c].dropna(), bins=edges, density=True)
        jsd_by_col[c] = float(jsd(hr, hs))
utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))






# 2) 안전성 검증(Single-Out Rate)
# 안전성 평가를 위한 행 매칭 키 생성
def make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=20):
    o, s = original.copy(), synth.copy()
    for c in num_cols:
        both = pd.concat([o[c], s[c]], axis=0)
        q = np.linspace(0, 1, qbins+1)
        edges = np.unique(np.nanquantile(both.dropna(), q))
        if len(edges) < 3:
            o[c+"_BIN"] = o[c].round(0).astype("Int64").astype("string")
            s[c+"_BIN"] = s[c].round(0).astype("Int64").astype("string")
        else:
            o[c+"_BIN"] = np.digitize(o[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
            s[c+"_BIN"] = np.digitize(s[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")

    match_cols = list(cat_cols) + [c+"_BIN" for c in num_cols]
    org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    return org_keys, syn_keys
# 합성 행 키가 원본 행 키 집합에 포함되는 비율(일치율) 계산.
def single_out_rate_binned(original, synth, cat_cols, num_cols, qbins=20):
    org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())
# 합성-원본 행 일치율 산출.
safety_single_out = single_out_rate_binned(df, synthetic, CAT_COLS, NUM_COLS, qbins=20)






In [ ]:
# @title
!pip install sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.3/197.3 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.5/14.5 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 198.3/198.3 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
# @title
# 혼합형(범주형 + 수치형) 테이블 합성: CTGAN (SDV 1.x)
# 1) 입력 데이터 컬럼 정의
# 2) 합성데이터 테이블 생성
# 3) 안전성 지표 점수 계산
# 4) 유용성 지표 점수 계산
# 5) 합성데이터 파일 저장

import os, json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import entropy

import sdv
print("SDV version:", sdv.__version__)

# --- (1) 입력 데이터 컬럼 정의 ----------------------------------------------------------
# 학습 컬럼 배분
CAT_COLS = ["연령대", "성별", "주택소재지(자치구)", "주거급여 수급여부", "국민임대 임대보증금액", "국민임대 월임대료"] # 범주형 컬럼 입력
NUM_COLS = ["가족수"] # 수치형 컬럼 입력
ALL_COLS = CAT_COLS + NUM_COLS

# 경로 설정
INPUT_XLSX = "/content/sample_data/3.국민임대 임대계약 정보 현황(합성용).xlsx" # 해당 경로의 폴더 내 원본데이터 저장 필요
OUTPUT_DIR = "/content/sample_data/" # 해당 경로의 폴더 내 합성데이터 저장 예정

# 하이퍼파라미터 조절
EPOCHS = 100
BATCH_SIZE = 64
PAC = 1

# 메타데이터 정의
from sdv.metadata import SingleTableMetadata
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(
    data=pd.DataFrame({c: pd.Series(dtype="object") for c in ALL_COLS})
)
for c in CAT_COLS:
    metadata.update_column(column_name=c, sdtype="categorical")
for c in NUM_COLS:
    metadata.update_column(column_name=c, sdtype="numerical")

# --- (2) 합성데이터 테이블 생성 ------------------------------------------
df = pd.read_excel(INPUT_XLSX, sheet_name=0)
df = df[ALL_COLS].copy()
# 전처리
for c in CAT_COLS:
    df[c] = df[c].astype("string")
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in NUM_COLS:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].median())
print("[LOAD] shape:", df.shape, "| dtypes:", {c: str(df[c].dtype) for c in df.columns})

# 모델링 학습
from sdv.single_table import CTGANSynthesizer

synth = CTGANSynthesizer(
    metadata,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    pac=PAC,
    verbose=True
)
synth.fit(df)

# 합성데이터 샘플링
synthetic = synth.sample(num_rows=1000) # 산출 행의 양을 설정
for c in CAT_COLS:
    synthetic[c] = synthetic[c].astype("string")
for c in NUM_COLS:
    synthetic[c] = pd.to_numeric(synthetic[c], errors="coerce")

# --- (3) & (4) 안전성, 유용성 지표 점수 계산 -------------------------------------------------
# 유용성 지표 계산 함수
def jsd(p, q, eps=1e-12):
    """Jensen-Shannon Divergence: 0에 가까울수록 분포 유사"""
    p = np.asarray(p, dtype=float) + eps
    q = np.asarray(q, dtype=float) + eps
    p = p / p.sum(); q = q / q.sum()
    m = 0.5*(p+q)
    return 0.5*(entropy(p, m) + entropy(q, m))

def make_matching_keys_with_binning(original: pd.DataFrame,
                                    synth: pd.DataFrame,
                                    cat_cols,
                                    num_cols,
                                    qbins: int = 20):
    """
    Single-Out Rate 계산을 위해:
      - 범주형은 그대로 사용
      - 수치형은 원본+합성의 분포를 기준으로 '공통 quantile bin'으로 이산화
        -> 수치형을 직접 값으로 비교하는 불공정(미세 소수점 차이 등) 방지
    """
    o = original.copy()
    s = synth.copy()

    for c in num_cols:
        both = pd.concat([o[c], s[c]], axis=0)
        try:
            q = np.linspace(0, 1, qbins+1)
            edges = np.unique(np.nanquantile(both.dropna(), q))
            if len(edges) < 3:
                o[c+"_BIN"] = o[c].round(0).astype("Int64").astype("string")
                s[c+"_BIN"] = s[c].round(0).astype("Int64").astype("string")
            else:
                o[c+"_BIN"] = np.digitize(o[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
                s[c+"_BIN"] = np.digitize(s[c], bins=edges[1:-1], right=True).astype("Int64").astype("string")
        except Exception:
            o[c+"_BIN"] = o[c].round(0).astype("Int64").astype("string")
            s[c+"_BIN"] = s[c].round(0).astype("Int64").astype("string")

    # 매칭에 사용할 최종 컬럼 묶음(범주형 + 수치형BIN)
    match_cols = list(cat_cols) + [c+"_BIN" for c in num_cols]

    org_keys = o[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    syn_keys = s[match_cols].astype(str).apply(lambda r: "||".join(r.values), axis=1)
    return org_keys, syn_keys
# 안전성 지표 계산 함수
def single_out_rate_binned(original: pd.DataFrame, synth: pd.DataFrame,
                           cat_cols, num_cols, qbins=20) -> float:
    """
    공통 quantile bin 기준으로 수치형을 이산화하여 공정하게 행 일치율 계산
    """
    org_keys, syn_keys = make_matching_keys_with_binning(original, synth, cat_cols, num_cols, qbins=qbins)
    org_set = set(org_keys.values)
    return float(syn_keys.isin(org_set).mean())

safety_single_out = single_out_rate_binned(df, synthetic, CAT_COLS, NUM_COLS, qbins=20)

jsd_by_col = {}

for c in CAT_COLS:
    vr = df[c].astype(str).value_counts(normalize=True)
    vs = synthetic[c].astype(str).value_counts(normalize=True)
    idx = vr.index.union(vs.index)
    jsd_by_col[c] = float(jsd(vr.reindex(idx, fill_value=0).values,
                              vs.reindex(idx, fill_value=0).values))

for c in NUM_COLS:
    both = pd.concat([df[c], synthetic[c]], axis=0).dropna()
    if both.nunique() <= 1:
        jsd_by_col[c] = 0.0
    else:
        bins = 20
        vmin, vmax = float(both.min()), float(both.max())
        edges = np.linspace(vmin, vmax, bins+1)
        hr, _ = np.histogram(df[c].dropna(), bins=edges, density=True)
        hs, _ = np.histogram(synthetic[c].dropna(), bins=edges, density=True)
        jsd_by_col[c] = float(jsd(hr, hs))

# 점수 결과 산출
utility_jsd_mean = float(np.mean(list(jsd_by_col.values())))
print(f"[SAFETY] Single-Out Rate (binned): {safety_single_out:.6f}")
print(f"[UTILITY] Mean JSD: {utility_jsd_mean:.6f}")
for k, v in jsd_by_col.items():
    print(f"  - {k}: {v:.6f}")

# --- (5) 합성데이터 파일 저장 ---------------------------------------------------------------
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
synthetic.to_csv(os.path.join(OUTPUT_DIR, "synthetic_data.csv"), index=False)
with pd.ExcelWriter(os.path.join(OUTPUT_DIR, "synthetic_data.xlsx")) as w:
    synthetic.to_excel(w, index=False)

report = {
    "config": {
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "pac": PAC,
        "cat_cols": CAT_COLS, "num_cols": NUM_COLS
    },
    "safety": {"single_out_rate_binned": safety_single_out, "qbins": 20},
    "utility": {"jsd_by_column": jsd_by_col, "jsd_mean": utility_jsd_mean},
    "notes": "혼합형 테이블용: 수치형은 binning 기반으로 안전성 매칭, 히스토그램 기반 JSD로 유용성 평가"
}
with open(os.path.join(OUTPUT_DIR, "evaluation_report.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print("Saved to:", OUTPUT_DIR)


SDV version: 1.30.0
[LOAD] shape: (1000, 7) | dtypes: {'연령대': 'string', '성별': 'string', '주택소재지(자치구)': 'string', '주거급여 수급여부': 'string', '국민임대 임대보증금액': 'string', '국민임대 월임대료': 'string', '가족수': 'int64'}


/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:168: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:134: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Gen. (-0.30) | Discrim. (-0.19): 100%|██████████| 100/100 [00:42<00:00,  2.37it/s]


[SAFETY] Single-Out Rate (binned): 0.137000
[UTILITY] Mean JSD: 0.003931
  - 연령대: 0.001972
  - 성별: 0.008639
  - 주택소재지(자치구): 0.005242
  - 주거급여 수급여부: 0.000000
  - 국민임대 임대보증금액: 0.002792
  - 국민임대 월임대료: 0.006960
  - 가족수: 0.001910
Saved to: /content/sample_data/
